<a href="https://colab.research.google.com/github/tuwadmin/eas_308_tutorial_6/blob/main/Copy_of_EAS308_tutorial6_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder in YOUR Google Drive where you saved the data file (see handout, Section 2)
path = "/content/drive/My Drive/EAS308"
!ls "{path}"

!pip install -q netCDF4 cartopy

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --- Open with xarray (decodes time units to datetime) ---
file_vis = os.path.join(path, "goes13.2011.117.200229.BAND_01.nc")
ds = xr.open_dataset(file_vis, decode_times=True)

# Identify the data variable and take the first slice (often time=0)
data_var = ds["data"]
vis_16bit = data_var.isel({data_var.dims[0]: 0})

# Convert 16-bit counts to 10-bit
vis_10bit = vis_16bit / 32.0

# Mask lon/lat edges (sentinel value)
mask_val = 2.14329e9
lon = ds["lon"].where(ds["lon"] != mask_val)
lat = ds["lat"].where(ds["lat"] != mask_val)
valid = lon.notnull() & lat.notnull()
vis_10bit = vis_10bit.where(valid)

# Grab valid time for title
vtime = ds["time"].values[0]   # np.datetime64

# --- Plot raw 10-bit counts on geostationary projection ---
mapcrs = ccrs.Geostationary(central_longitude=-75)
fig, ax = plt.subplots(figsize=(19, 10), subplot_kw={"projection": mapcrs})

# Use pcolormesh with geolocation + transform; no need to compute extent
pm = ax.pcolormesh(
    lon, lat, vis_10bit,
    transform=ccrs.PlateCarree(),
    cmap="Greys_r",
    shading="auto"
)
cb = fig.colorbar(pm, ax=ax, orientation="vertical", label="10-bit Visible Counts")

# Map features
ax.add_feature(cfeature.COASTLINE.with_scale("50m"), edgecolor="yellow", linewidth=0.5)
ax.add_feature(cfeature.STATES.with_scale("50m"),    edgecolor="yellow", linewidth=0.5)

plt.title("GOES-13 Visible Satellite Imagery (Raw Counts)", loc="left")
plt.title(np.datetime_as_string(vtime, unit="s"),         loc="right")

plt.tight_layout()
plt.show()

In [ ]:
# --- Calibration: counts -> radiance -> effective albedo ---
# GOES-13 visible (channel 1) coefficients
m_cal = 6.120196e-1
b_cal = -17.749
k_cal = 1.89544e-3

R_vis = m_cal * vis_10bit + b_cal    # scene radiance
albedo = k_cal * R_vis               # effective albedo (0 = dark, 1 = bright)

# --- Plot effective albedo on geostationary projection ---
fig, ax = plt.subplots(figsize=(19, 10), subplot_kw={"projection": mapcrs})

pm = ax.pcolormesh(
    lon, lat, albedo,
    transform=ccrs.PlateCarree(),
    cmap="Greys_r",
    vmin=0, vmax=1,
    shading="auto"
)
cb = fig.colorbar(pm, ax=ax, orientation="vertical", label="Effective Albedo")

# Map features
ax.add_feature(cfeature.COASTLINE.with_scale("50m"), edgecolor="yellow", linewidth=0.5)
ax.add_feature(cfeature.STATES.with_scale("50m"),    edgecolor="yellow", linewidth=0.5)

plt.title("GOES-13 Visible Satellite Imagery (Effective Albedo)", loc="left")
plt.title(np.datetime_as_string(vtime, unit="s"),               loc="right")

plt.tight_layout()
plt.show()